### Projeto Anac
![](https://www.gov.br/anac/pt-br/banner-rotativo/antigos/logo-anac.png)

In [0]:
from pyspark.sql.functions import col, trim

In [0]:
df = spark.read.json("/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.json")
display(df)

In [0]:
df = df.na.drop()

In [0]:
df = df.filter(~(
    (trim(col("Aerodromo_de_Destino")) == "") |
    (trim(col("Aerodromo_de_Origem")) == "") |
    (trim(col("CLS")) == "") |
    (trim(col("Categoria_da_Aeronave")) == "")
))

In [0]:
# Filter Text 
df_SP = df.filter(df.UF == "SP")
display(df_SP)

In [0]:
# Acidentes graves região sudeste
display(df.filter((col("UF") == "SP") & (col("Fase_da_Operacao") == "Decolagem")))

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, year, month, when, regexp_replace, lower, trim, avg, max, min
)

spark = SparkSession.builder.appName("AnaliseAcidentesAereos").getOrCreate()

# Exemplo: leitura do CSV
df_raw = spark.read.option("header", True).option("sep", "\t").json("/Volumes/workspace/default/arquivos-aula/Anac/V_OCORRENCIA_AMPLA.json")

### impeza e padronização (Transform)

In [0]:
# Remover linhas totalmente nulas
df = df_raw.na.drop("all")

# Limpar espaços e padronizar colunas de texto
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# Converter campos numéricos de lesões
campos_lesoes = [
    "Ilesos_Passageiros", "Ilesos_Tripulantes",
    "Lesoes_Fatais_Passageiros", "Lesoes_Fatais_Tripulantes",
    "Lesoes_Graves_Passageiros", "Lesoes_Graves_Tripulantes",
    "Lesoes_Leves_Passageiros", "Lesoes_Leves_Tripulantes"
]

for c in campos_lesoes:
    df = df.withColumn(c, col(c).cast("int"))

# Corrigir latitude/longitude (vírgula → ponto)
df = df.withColumn("Latitude", regexp_replace("Latitude", ",", ".").cast("double"))
df = df.withColumn("Longitude", regexp_replace("Longitude", ",", ".").cast("double"))

# Converter data
df = df.withColumn("Data_da_Ocorrencia", col("Data_da_Ocorrencia").cast("date"))

### Enriquecimento de dados

In [0]:
df = df.withColumn("Ano", year("Data_da_Ocorrencia"))
df = df.withColumn("Mes", month("Data_da_Ocorrencia"))

# Criar coluna "Gravidade" baseada nos danos e tipo
df = df.withColumn(
    "Nivel_Gravidade",
    when(col("Danos_a_Aeronave") == "Substancial", "Alta")
    .when(col("Classificacao_da_Ocorrência").like("%Grave%"), "Média")
    .otherwise("Baixa")
)

### Seleção das colunas principais

In [0]:
df_analise = df.select(
    "Data_da_Ocorrencia",
    "Ano", "Mes", "UF", "Municipio", "Modelo", "Nome_do_Fabricante",
    "Fase_da_Operacao", "Descricao_do_Tipo",
    "Classificacao_da_Ocorrência", "Danos_a_Aeronave",
    "Operacao", "Operador_Padronizado",
    "Ilesos_Passageiros", "Lesoes_Fatais_Passageiros",
    "Lesoes_Graves_Passageiros", "Lesoes_Leves_Passageiros",
    "Nivel_Gravidade"
)

### Estatísticas descritivas e agregações

In [0]:
ocorrencias_ano = (
    df_analise.groupBy("Ano", "Nivel_Gravidade")
    .agg(count("*").alias("Total_Ocorrencias"))
    .orderBy("Ano")
)
display(ocorrencias_ano)

### Top 10 causas mais comuns

In [0]:
causas = (
    df_analise.groupBy("Descricao_do_Tipo")
    .agg(count("*").alias("Qtde"))
    .orderBy(col("Qtde").desc())
)
display(causas.limit(10))

### Distribuição por fase de operação

In [0]:
fase_operacao = (
    df_analise.groupBy("Fase_da_Operacao")
    .agg(count("*").alias("Total"))
    .orderBy(col("Total").desc())
)
display(fase_operacao)

### Lesões médias por tipo de operação

In [0]:
lesoes = (
    df_analise.groupBy("Operacao")
    .agg(
        avg("Lesoes_Fatais_Passageiros").alias("Media_Letais"),
        avg("Lesoes_Graves_Passageiros").alias("Media_Graves"),
        avg("Lesoes_Leves_Passageiros").alias("Media_Leves")
    )
)
display(lesoes)

In [0]:
from pyspark.sql.functions import year, month, col

# Converter Data_da_Ocorrencia para tipo date (caso ainda esteja como string)
df = df.withColumn("Data_da_Ocorrencia", col("Data_da_Ocorrencia").cast("date"))

# Criar colunas de ano e mês
df = df.withColumn("Ano", year(col("Data_da_Ocorrencia")))
df = df.withColumn("Mes", month(col("Data_da_Ocorrencia")))

# Converter para Pandas para usar com Plotly
df_pd = df.toPandas()

### Padrões temporais – Ocorrências por mês/ano

In [0]:
import plotly.express as px

ocorrencias_tempo = (
    df_pd.groupby(["Ano", "Mes"])
    .size()
    .reset_index(name="Total_Ocorrencias")
    .sort_values(["Ano", "Mes"])
)

fig1 = px.line(
    ocorrencias_tempo,
    x="Mes",
    y="Total_Ocorrencias",
    color="Ano",
    markers=True,
    title="📅 Ocorrências por Mês e Ano",
    labels={"Mes": "Mês", "Total_Ocorrencias": "Total de Ocorrências"}
)
fig1.show()

### Análise de causas – Top 10 tipos de ocorrência

In [0]:
import plotly.express as px

# Contagem dos tipos de ocorrência
causas = (
    df_pd["Descricao_do_Tipo"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "Tipo_de_Ocorrencia", "Descricao_do_Tipo": "Total"})
    .head(10)
)

# Corrigir nome das colunas se o rename anterior não funcionar
if "Tipo_de_Ocorrencia" not in causas.columns:
    causas.columns = ["Tipo_de_Ocorrencia", "Total"]

# Gerar gráfico
fig2 = px.bar(
    causas,
    x="Tipo_de_Ocorrencia",
    y="Total",
    title="Top 10 Causas de Ocorrências",
    color="Total",
    text="Total"
)
fig2.update_layout(xaxis_tickangle=60)
fig2.show()

### Análise geoespacial

In [0]:
%pip install folium

### Análise geoespacial – Ocorrências no mapa (Folium)

In [0]:
import folium
from folium.plugins import MarkerCluster

# Filtra registros válidos com coordenadas
df_geo = df_pd.dropna(subset=["Latitude", "Longitude"])

m = folium.Map(location=[-15.78, -47.93], zoom_start=4, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)

for _, row in df_geo.iterrows():
    popup_text = f"""
    <b>Município:</b> {row['Municipio']}<br>
    <b>Operação:</b> {row['Operacao']}<br>
    <b>Fase:</b> {row['Fase_da_Operacao']}<br>
    <b>Tipo:</b> {row['Descricao_do_Tipo']}
    """
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=popup_text
    ).add_to(marker_cluster)

m

In [0]:
import folium
from folium.plugins import MarkerCluster

# Exemplo simples de mapa
mapa = folium.Map(location=[-15.78, -47.93], zoom_start=4)  # Brasil
folium.Marker(location=[-23.55, -46.63], popup="São Paulo").add_to(mapa)
display(mapa)

In [0]:
# Remove linhas com latitude ou longitude nulas
df_analise = df_analise.filter(
    (df_analise["latitude"].isNotNull()) &
    (df_analise["longitude"].isNotNull())
)

# Remove linhas onde 'Operacao' está nula (se for usada no popup ou cor)
df_analise = df_analise.filter(df_analise["Operacao"].isNotNull())

display(df_analise)

### Modelos de risco – Ocorrências por operação e fabricante

In [0]:
import plotly.express as px
from pyspark.sql import functions as F
import pandas as pd

# ============================================================
# Agrupar e preparar dados
# ============================================================
df_risco = (
    df_SP.groupBy("Fase_da_Operacao", "Nome_do_Fabricante")
    .count()
    .orderBy(F.desc("count"))
)

pdf_risco = (
    df_risco.toPandas()
    .dropna(subset=["Fase_da_Operacao", "Nome_do_Fabricante"])
)

# Reordenar categorias para destacar as mais frequentes
top_fases = pdf_risco.groupby("Fase_da_Operacao")["count"].sum().sort_values(ascending=False).index
top_fabricantes = pdf_risco.groupby("Nome_do_Fabricante")["count"].sum().sort_values(ascending=False).index
pdf_risco["Fase_da_Operacao"] = pd.Categorical(pdf_risco["Fase_da_Operacao"], categories=top_fases, ordered=True)
pdf_risco["Nome_do_Fabricante"] = pd.Categorical(pdf_risco["Nome_do_Fabricante"], categories=top_fabricantes, ordered=True)

# ============================================================
# Criar gráfico de calor avançado
# ============================================================
fig_risco = px.density_heatmap(
    pdf_risco,
    x="Fase_da_Operacao",
    y="Nome_do_Fabricante",
    z="count",
    color_continuous_scale="Viridis",
    title="🛫 Mapa de Risco: Ocorrências por Fase da Operação e Fabricante",
    labels={
        "count": "Quantidade de Ocorrências",
        "Fase_da_Operacao": "Fase da Operação",
        "Nome_do_Fabricante": "Fabricante"
    },
    text_auto=True
)

# ============================================================
# Ajustes visuais
# ============================================================
fig_risco.update_layout(
    title_font_size=22,
    title_x=0.5,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    coloraxis_colorbar_title="Ocorrências",
    width=950,
    height=700,
    xaxis={'categoryorder': 'array', 'categoryarray': list(top_fases)},
    yaxis={'categoryorder': 'array', 'categoryarray': list(top_fabricantes)},
)

# Mostrar gráfico
fig_risco.show()

In [0]:
# Imports necessários
import pandas as pd
import plotly.express as px
from pyspark.sql import functions as F

# 1) Detectar colunas com fallback
cols = df_SP.columns
aircraft_col = None
severity_col = None

# Possíveis nomes que sua base usa
candidates_aircraft = ["Tipo_de_Aeronave", "Categoria_da_Aeronave", "Modelo"]
candidates_severity = ["Nivel_de_Dano", "Danos_a_Aeronave", "Classificacao_da_Ocorrência"]

for c in candidates_aircraft:
    if c in cols:
        aircraft_col = c
        break

for c in candidates_severity:
    if c in cols:
        severity_col = c
        break

if aircraft_col is None or severity_col is None:
    raise ValueError(f"Colunas esperadas não encontradas. Encontradas: {cols}")

# 2) Agrupar e contar ocorrências
df_gravidade = (
    df_SP
    .select(aircraft_col, severity_col)
    .filter(F.col(aircraft_col).isNotNull() & F.col(severity_col).isNotNull())
    .groupBy(aircraft_col, severity_col)
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

# 3) Converter para pandas com segurança (limitar linhas para não estourar memória)
pdf = df_gravidade.toPandas()

# 4) Se houver muitas categorias, manter apenas top N de aeronaves para visualização
TOP_N = 20
top_aircraft = pdf.groupby(aircraft_col)["count"].sum().nlargest(TOP_N).index.tolist()
pdf = pdf[pdf[aircraft_col].isin(top_aircraft)].copy()

# 5) Tornar as colunas categóricas ordenadas para legibilidade
pdf[aircraft_col] = pd.Categorical(pdf[aircraft_col], categories=top_aircraft, ordered=True)
# Ordenar níveis de gravidade por frequência (opcional)
severity_order = pdf.groupby(severity_col)["count"].sum().sort_values(ascending=False).index.tolist()
pdf[severity_col] = pd.Categorical(pdf[severity_col], categories=severity_order, ordered=True)

# 6) Plot: scatter de bolhas (cada bolha = combinação aeronave x gravidade)
fig_gravidade = px.scatter(
    pdf,
    x=aircraft_col,
    y=severity_col,
    size="count",
    color="count",                       # cor pela quantidade (poderia ser severity_col também)
    hover_name=aircraft_col,
    hover_data={severity_col: True, "count": True},
    title="✈️ Correlação entre Gravidade (Danos) e Tipo/Categoria de Aeronave",
    labels={"count": "Número de Ocorrências", aircraft_col: "Tipo/Categoria de Aeronave", severity_col: "Gravidade / Danos"}
)

# 7) Ajustes visuais
fig_gravidade.update_layout(
    xaxis_tickangle=-45,
    width=1100,
    height=600,
    title_x=0.5,
    showlegend=False
)

# 8) Se quiser ordenar o eixo Y de maneira legível e centralizar o texto de hover:
fig_gravidade.update_traces(marker=dict(sizemode='area', sizeref=pdf['count'].max()/200.0))

# Mostrar o gráfico
fig_gravidade.show()